# Powerlifting Performances

This notebook addresses a regression challenge based on public OpenPowerlifting data, the largest collection of powerlifting competition results, updated daily and in the public domain. Each record describes an athlete's performance at a full meet along with athlete, federation, and competition characteristics.

The goal is to predict the achieved total **(TotalKg)** from 16 explanatory variables: three numerical and thirteen categorical. The training set contains roughly 105,000 labeled rows, and the test set about 45,000.

Performance is measured with **RMSE** (Root Mean Square Error), so predictions are evaluated by how closely they match the achieved total in kilograms.

## 1. Imports

In [1]:
# Imports
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split

from regress_utils import*

# Global Config
warnings.filterwarnings("ignore")
TRAIN_PATH, TEST_PATH = "data_reg/train.csv", "data_reg/test.csv"
MODEL_SIZE = "small"

print("Device:", DEVICE)

Device: cuda


## 2. Data

An 80/20 random train/validation split is used. All preprocessing steps: feature engineering, imputation, scaling, and encoding—are fitted exclusively on the training rows. Two feature representations are then constructed:

* **scl:** imputed, scaled, and one-hot encoded features, used for linear models, tree-based methods, boosting, and MLPs.
* **ord:** cleaned numeric features that retain missing values, together with ordinally encoded categoricals, tailored to foundation models.

## 2.1 Train

In [2]:
# Load train data
df = pd.read_csv(TRAIN_PATH)
y = df.pop(TARGET).to_numpy(np.float64)
df.pop("id")

# Split and preprocess
i_tr, i_val = train_test_split(np.arange(len(df)), test_size=0.2, random_state=SEED)
prep = Preprocessor().fit(df.iloc[i_tr])
X = prep.transform_views(df)
X_tr, X_val = take(X, i_tr), take(X, i_val)
y_tr, y_val = y[i_tr], y[i_val]

# Summary target
print(f"{TARGET}: mean {y.mean():.1f} kg | std {y.std():.1f} kg | min {y.min():.1f} | max {y.max():.1f}")

TotalKg: mean 493.0 kg | std 171.4 kg | min 24.0 | max 1363.0


## 2.2 Test

In [3]:
# Load and preprocess test data
test = pd.read_csv(TEST_PATH)
test_ids = test.pop("id")
test = test.drop(columns=[TARGET], errors="ignore")
X_test = prep.transform_views(test)

## 3. Model Training
Each model is fitted on the training set, its validation probabilities are stored in **P_val**, and the fitted estimator is kept in **models**.

In [4]:
models, P_val = {}, {}

### 3.1 Baseline
Predicts the training class prior for every row and serves as the reference for all other models.

In [5]:
# Define Model
models["Baseline"] = DummyRegressor(strategy="mean")

# Fit and Predict
P_val["Baseline"] = fit_predict(models["Baseline"], X_tr, y_tr, X_val)

### 3.2 Ridge Regression
Linear regression with L2 regularization, which shrinks coefficients to stabilize estimates.

In [6]:
# Define Model
models["Ridge"] = Ridge(alpha=1.0)

# Fit and Predict
P_val["Ridge"] = fit_predict(models["Ridge"], X_tr, y_tr, X_val)

### 3.3 Random Forest
A bagging ensemble of 1,000 decision trees, each trained on a bootstrap sample with random feature subsets per split.

In [7]:
# Define Model
models["RandomForest"] = RandomForestRegressor(n_estimators=1000, min_samples_leaf=3, max_features=0.33,
                                                n_jobs=-1, random_state=SEED)
# Fit and Predict
P_val["RandomForest"] = fit_predict(models["RandomForest"], X_tr, y_tr, X_val)

### 3.4 Extra Trees
An ensemble of 300 extremely randomized trees: unlike Random Forest, split thresholds are drawn at random and no bootstrap sampling is used.

In [8]:
# Define Model
models["ExtraTrees"] = ExtraTreesRegressor(n_estimators=300, min_samples_leaf=3, max_features=0.5,
                                            n_jobs=-1, random_state=SEED)
# Fit and Predict
P_val["ExtraTrees"] = fit_predict(models["ExtraTrees"], X_tr, y_tr, X_val)

### 3.5 Histogram Gradient Boosting
Gradient-boosted trees trained on histogram-binned features. Predictions average three seeds, each using early stopping.

In [9]:
# Define hyperparameters
HGB_PARAMS = dict(learning_rate=0.05, max_leaf_nodes=63, min_samples_leaf=20, l2_regularization=1.0,
                  max_iter=2000, early_stopping=True, validation_fraction=0.1, n_iter_no_change=50)

# Define Model
models["HistGB"] = SeedBag(HGB_PARAMS, n_seeds=3)

# Fit and Predict
P_val["HistGB"] = fit_predict(models["HistGB"], X_tr, y_tr, X_val)

### 3.6 XGBoost
Gradient-boosted trees using a second-order loss approximation, histogram-based splits, and leaf-wise growth.

In [10]:
# Define Model
models["XGBoost"] = EarlyStopped(
    XGBRegressor(n_estimators=2000, learning_rate=0.03, tree_method="hist", grow_policy="lossguide",
                 max_leaves=63, min_child_weight=5, colsample_bynode=0.7, reg_lambda=5.0, eval_metric="rmse",
                 early_stopping_rounds=100, n_jobs=-1, random_state=SEED),
    fit_kw=lambda: dict(verbose=False))

# Fit and Predict
P_val["XGBoost"] = fit_predict(models["XGBoost"], X_tr, y_tr, X_val)

### 3.7 LightGBM
A gradient boosting framework that builds trees leaf-wise and uses histogram-binned features, making it fast and memory-efficient on large tabular datasets.

In [11]:
# Define Model
models["LightGBM"] = EarlyStopped(
    LGBMRegressor(n_estimators=2000, learning_rate=0.05, num_leaves=63, min_child_samples=20,
                  feature_fraction_bynode=0.7, reg_lambda=5.0, n_jobs=-1, random_state=SEED, verbose=-1),
    fit_kw=lambda: dict(eval_metric="rmse", callbacks=[lgb.early_stopping(100, verbose=False)]))

# Fit and Predict
P_val["LightGBM"] = fit_predict(models["LightGBM"], X_tr, y_tr, X_val)

### 3.8 Multilayer Perceptron 
A feedforward neural network with two hidden layers, each combining a linear transformation, batch normalization, sigmoid activation, and dropout. It is trained with Adam and binary cross-entropy.

In [12]:
# Define Model
models["MLP"] = TorchMLP(hidden=(128, 64), dropout=0.1, lr=1e-3, batch=256, max_epochs=100, patience=5)

# Fit and Predict
P_val["MLP"] = fit_predict(models["MLP"], X_tr, y_tr, X_val)
print("Best epoch:", models["MLP"].best_epoch_)

  0%|          | 0/100 [00:00<?, ?epoch/s]

Best epoch: 41


## 4. Foundation Models
Foundation models are pretrained tabular transformers used here as in-context learners: no gradient updates are performed on the dataset. Instead, labeled training rows serve as context, and predictions are obtained through a forward pass. These models consume the **ord** feature view, with the context capped at **max_context** rows and queries scored in batches.

In [13]:
# Small batches
s_batch = dict(cat_idx=prep.ord_cat_idx, batch_rows=10000)

# Big batches
b_batch = dict(cat_idx=prep.ord_cat_idx, batch_rows=100000)

### 4.1 TabPFN-3.5
A Prior Labs transformer pretrained on synthetic tabular datasets, designed to approximate the Bayesian posterior predictive in a single forward pass.

In [14]:
# Define Model and Predict
run_model("TabPFN-3.5", TabPFNModel(version="3.5", size=MODEL_SIZE, max_context=10000, **s_batch),
          models, P_val, X_tr, y_tr, X_val)

### 4.2 TabICLv2
A transformer for tabular in-context learning, pretrained on synthetic datasets. It embeds features column-wise, models interactions across rows, and then performs in-context prediction.

In [15]:
# Define Model and Predict
run_model("TabICLv2", TabICLModel(n_estimators=4, max_context=25000, **s_batch),
          models, P_val, X_tr, y_tr, X_val)

### 4.3 Kumo-Tabular
A NVIDIA foundation model for tabular data. It outputs multiple quantiles, and the final prediction is the mean of these quantiles.

In [16]:
# Define Model and Predict
run_model("Kumo-Tabular", KumoTabularModel(size=MODEL_SIZE, max_context=None, **b_batch),
          models, P_val, X_tr, y_tr, X_val)

## 5 Results
Validation metrics are reported below, sorted by RMSE. MedAE denotes the median absolute error, and MAPE the mean absolute percentage error.

In [17]:
# Results Table
results = results_table(P_val, y_val)
results

,RMSE,MAE,MedAE,R2,MAPE %
Kumo-Tabular,77.5731,59.4253,47.5050,0.7944,13.4516
HistGB,77.9608,59.7931,47.7511,0.7924,13.5722
XGBoost,78.1498,59.8568,47.5524,0.7913,13.5707
LightGBM,78.1857,59.9580,47.8275,0.7912,13.6056
MLP,80.3530,61.7806,49.3583,0.7794,14.1258
TabICLv2,80.5951,61.8659,49.5358,0.7781,14.0762
TabPFN-3.5,80.8812,61.8485,49.4031,0.7765,14.1070
RandomForest,81.2372,62.1866,49.7970,0.7745,14.1140
ExtraTrees,82.2728,62.9654,50.1625,0.7688,14.2685
Ridge,85.6869,66.1265,53.2599,0.7492,15.2594


## 6. Test Predictions
The model with the lowest validation RMSE is retrained on the combined training and validation data, then used to generate predictions on the test set.

In [18]:
# Best model
best = results["RMSE"].idxmin()
print(f"Selected model: {best} | validation RMSE: {results.loc[best, 'RMSE']:.2f} kg | R2: {results.loc[best, 'R2']:.4f}")

# Refit and Predict
p_test = refit_predict(models, best, stack_views(X_tr, X_val), np.concatenate([y_tr, y_val]), X_test)
pred = pd.DataFrame({"id": test_ids, TARGET: np.clip(p_test, 0, None).round(3)})

# Save Submission
pred.to_csv("regress_pred.csv", index=False)
print(f"Saved regress_pred.csv: {len(pred):,} rows | mean predicted {TARGET}: {pred[TARGET].mean():.1f} kg")

# Preview
pred.head()

Selected model: Kumo-Tabular | validation RMSE: 77.57 kg | R2: 0.7944
Saved regress_pred.csv: 45,000 rows | mean predicted TotalKg: 494.2 kg


,id,TotalKg
0,59770,768.962
1,21362,525.346
2,127324,649.107
3,140509,418.447
4,144297,307.553
